In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("========== STEP 1: Read parameters ==========")
print(f"Landing timestamp : {landing_timestamp}")
print(f"Table name        : {table_name}")
print(f"Merge keys        : {merge_keys}")
print(f"Load type         : {load_type}")

In [0]:
print("========== STEP 2: Read data from Bronze ==========")

source_table = f"adb_mediflow.bronze.{table_name}"
filter_condition = f"landing_timestamp = '{landing_timestamp}'"

print(f"Source table      : {source_table}")
print(f"Filter condition  : {filter_condition}")
print(f"Equivalent query  : SELECT * FROM {source_table} WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

print(f"Rows read from Bronze : {bronze_df.count()}")

In [0]:
from pyspark.sql.functions import current_timestamp
updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp())\
    .withColumn("update_timestamp", current_timestamp())

print("========== STEP 3: Add audit columns ==========")
print("Added columns     : insert_timestamp, update_timestamp")
print(f"Final columns     : {updated_df.columns}")

In [0]:
target_table = f"adb_mediflow.silver.{table_name}"

print("========== STEP 4: Write to Silver ==========")
print(f"Target table      : {target_table}")
print(f"Load type         : {load_type}")

if load_type == "FULL":
    print(f"FULL load -> overwriting {target_table}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("FULL load completed")

elif load_type == "APPEND":
    print(f"APPEND load -> appending data to {target_table}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("APPEND load completed")

elif load_type == "MERGE" and not spark.catalog.tableExists(target_table):
    # table not there yet, so do a full load first
    print(f"MERGE load -> {target_table} does not exist, doing a FULL load to create it")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("Table created with FULL load")

elif load_type == "MERGE":
    print(f"MERGE load -> {target_table} exists, merging data")

    updated_df.createOrReplaceTempView("source")
    print("Created temp view 'source' from the new data")

    # e.g. merge_keys = "id,code"  ->  t.id = s.id AND t.code = s.code
    merge_condition = " AND ".join([f"t.{key} = s.{key}" for key in merge_keys.split(",")])
    print(f"Merge condition   : {merge_condition}")

    merge_query = f"""
        MERGE INTO {target_table} t
        USING source s
        ON {merge_condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """
    print("Merge query to be executed:")
    print(merge_query)

    merge_result = spark.sql(merge_query)
    print("MERGE completed. Result (rows affected / updated / deleted / inserted):")
    merge_result.show()

else:
    print(f"Unknown load type '{load_type}', nothing was written")

In [0]:
print("========== STEP 5: Return record count ==========")

record_count = updated_df.count()
print(f"Records processed : {record_count}")
print("Bronze to Silver completed, exiting notebook")

dbutils.notebook.exit(str(record_count))